In [2]:
import pandas as pd
import numpy as np
import random
import warnings
from datetime import datetime



# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
price_data = pd.read_csv('E:\Signal Backtesting\Input\price_2023-06-01_to_2024-08-4_min.csv', parse_dates=['Datetime'],
                         index_col='Datetime')
# Re-load the signal data without setting the index
signal_data = pd.read_excel('E:\Signal Backtesting\Input\\value.xlsx',
                          parse_dates=['Datetime'])
#month = 5  # January (you can change this to the desired month)
# year = 2024  # You can change this to the desired year
# # # 
# # # # Filter the signal data for the specified month and year
# signal_data = signal_data[ (signal_data['Datetime'].dt.year == year)]

In [35]:
# Backtest trades function
def backtest_trades(price_data, signal_data, tp=None, sl=None, entry_time_offset=None,
                    percentage_change=None, open_order_elimination=None, ignore_time_interval_before=None,
                    ignore_time_interval_after=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Signal Open Price', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration',
        'Execution Latency', 'ROI', 'NAV', 'Ignore Reason'
    ])

    initial_margin = 100000
    current_margin = initial_margin
    exit_datetimes = []
    initial_drawdown = 0  # For initial drawdown calculation
    nav_history = []

    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']

        if signal_value == 0:
            continue
        elif signal_value > 0:
            side = 'Buy'
        else:
            side = 'Sell'

        adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
        signal_open_price = price_data.at[adjusted_signal_datetime, 'Open']

        # Ignoring signals based on open trades
        exit_datetimes.sort(key=lambda x: x[0])
        ignore_signal = False
        reason = ''

        if exit_datetimes:
            later_exits = [ed for ed in exit_datetimes if ed[0] > signal_datetime]

            if len(later_exits) >= 2:
                result = 'Ignored'
                reason = '2 open trades'
                ignore_signal = True
            elif len(later_exits) == 1:
                if later_exits[-1][1] != side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'One open trade with the same side'
                    ignore_signal = True

        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': result,
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': reason
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        # New logic: ignore signals within a specific time interval before and after events
        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            ignore_signal = any(event_datetime - pd.Timedelta(
                minutes=ignore_time_interval_before) <= signal_datetime <= event_datetime + pd.Timedelta(
                minutes=ignore_time_interval_after)
                                for event_datetime in event_times)
        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Ignored',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': 'Signal around economic event'
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        entry_datetime, entry_price, entry_duration = determine_entry(price_data, signal_datetime, percentage_change,
                                                                      side, open_order_elimination, entry_time_offset)
        if entry_datetime is None:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Not Filled',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': ''
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        # Update NAV on filled order
        current_margin *= (1 - 0.0002)

        if side == 'Buy':
            tp_price = entry_price * (1 + tp)
            sl_price = entry_price * (1 - sl)
        else:
            tp_price = entry_price * (1 - tp)
            sl_price = entry_price * (1 + sl)

        result, duration_str = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)
        exit_datetime = entry_datetime + pd.Timedelta(duration_str)

        if result in [1, -1]:
            exit_datetimes.append((exit_datetime, side))

        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            for event_datetime in event_times:
                if entry_datetime < event_datetime < exit_datetime:
                    exit_datetime = event_datetime - pd.Timedelta(minutes=10)
                    if exit_datetime in price_data.index:
                        exit_price = price_data.at[exit_datetime, 'Open']
                        result = 'ended before data'
                        if (side == 'Buy' and exit_price > entry_price) or (
                                side == 'Sell' and exit_price < entry_price):
                            result += ' with profit'
                            pct_change = (exit_price - entry_price) / entry_price if side == 'Buy' else (
                                entry_price - exit_price) / entry_price
                            current_margin = current_margin * (1 + pct_change)
                        else:
                            result += ' with loss'
                            pct_change = (entry_price - exit_price) / entry_price if side == 'Buy' else (
                                exit_price - entry_price) / entry_price
                            current_margin = current_margin * (1 - pct_change)
                    break

        if result not in ['ended before data with profit', 'ended before data with loss',
                          'ended before data with no exact price']:

            if result == 1:
                current_margin = current_margin * (1 + tp)
                current_margin *= (1 - 0.0005)
            elif result == -1:
                current_margin = current_margin * (1 - sl)
                current_margin *= (1 - 0.0005)

        # Calculate initial drawdown
        if current_margin < 100000:
            drawdown = ((100000 - current_margin) / 100000) * 100
            initial_drawdown = max(initial_drawdown, drawdown)
        
        roi = ((current_margin - initial_margin) / initial_margin) * 100
        nav = current_margin       
        initial_margin = current_margin

        nav_history.append(nav)
        new_row = pd.DataFrame([{
            'Datetime': signal_datetime,
            'Side': side,
            'Signal Open Price': signal_open_price,
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Result': result,
            'Duration': duration_str,
            'Execution Latency': format_duration(entry_duration),
            'ROI': roi,
            'NAV': nav,
            'Ignore Reason': ''
        }])

        output_data = pd.concat([output_data, new_row], ignore_index=True)
  
    # Ensure 'Datetime' column is in datetime format
    output_data['Datetime'] = pd.to_datetime(output_data['Datetime'])

    # Calculate Daily Return
    output_data['Date'] = output_data['Datetime'].dt.date
    daily_nav = output_data.groupby('Date')['NAV'].last().to_dict()
    daily_returns = {}
    previous_day_nav = 100000

    for date, nav in daily_nav.items():
        daily_return = ((nav - previous_day_nav) / previous_day_nav) * 100
        daily_returns[date] = daily_return
        previous_day_nav = nav

    output_data['Daily Return'] = output_data['Date'].map(daily_returns)
    output_data.drop(columns=['Date'], inplace=True)
    
    # Calculate Monthly Max Drawdown
    output_data['Month'] = output_data['Datetime'].dt.to_period('M')
    monthly_max_drawdowns = {}

    for month, group in output_data.groupby('Month'):
        peak_nav = group['NAV'].iloc[0]  # Start with the first NAV of the month
        max_drawdown_in_month = 0
        local_peak = peak_nav

        for nav in group['NAV']:
            if nav > local_peak:
                local_peak = nav  # Update the peak if a new high is found
            else:
                # Calculate drawdown from the peak to the current NAV
                drawdown = ((local_peak - nav) / local_peak) * 100
                max_drawdown_in_month = max(max_drawdown_in_month, drawdown)  # Track the maximum drawdown

        monthly_max_drawdowns[month] = max_drawdown_in_month


    output_data['Monthly Max Drawdown'] = output_data['Month'].map(monthly_max_drawdowns)
    output_data['Initial Drawdown'] = initial_drawdown
   
    output_data.drop(columns=['Month'], inplace=True)
    
    return output_data

# Helper functions
def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"

def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
    
    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'
        
    return result, duration_str

def determine_entry(price_data, signal_datetime, percentage_change, side, open_order_elimination, entry_time_offset):
    adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
    if adjusted_signal_datetime not in price_data.index:
        return None, None, None
    
    adjusted_open_price = price_data.at[adjusted_signal_datetime, 'Open']
    percentage_change_price = adjusted_open_price * (1 - percentage_change) if side == 'Buy' else adjusted_open_price * (1 + percentage_change)
    
    time_limit = adjusted_signal_datetime + pd.Timedelta(minutes=open_order_elimination)
    subsequent_prices = price_data.loc[adjusted_signal_datetime:time_limit]
    
    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration

    return None, None, None

In [36]:
def filter_signals(signal_data):
    filtered_signals = []
    seen_signal = False

    for i, row in signal_data.iterrows():
        signal_value = row['Signal']

        if signal_value == 0:
            seen_signal = False
            filtered_signals.append(row)
            continue

        if not seen_signal:
            if signal_value == 1 or signal_value == -1:
                seen_signal = True
                filtered_signals.append(row)
        else:
            # Modify the row to reset the signal to 0
            row['Signal'] = 0
            filtered_signals.append(row)

    return pd.DataFrame(filtered_signals)

In [37]:
# Define the parameter ranges
tp_values = np.arange(0.009, 0.014, 0.001)
sl_values = np.arange(0.009, 0.014, 0.001)
entry_time_offset_values = np.arange(0, 120, 10)
percentage_change_values = np.arange(0.0001, 0.0015, 0.0001)
ignore_time_interval_before= np.arange(0,1080,60)

In [31]:
def create_individual():
    rolling_window = np.random.choice(range(5, 49))  # Rolling window from 5H to 48H
    upper_std_multiplier = np.random.choice(np.arange(0.7, 2.1, 0.1))  # Std multipliers from 0.7 to 3 by 0.1
    lower_std_multiplier = np.random.choice(np.arange(0.7, 2.1, 0.1))  # Separate range for lower multiplier
    while True:
        tp = np.random.choice(tp_values)
        sl = np.random.choice(sl_values)
        if sl < tp:  # Ensure that SL is less than TP
            break
    return [
        rolling_window,
        upper_std_multiplier,
        lower_std_multiplier,
        tp,
        sl,
        np.random.choice(entry_time_offset_values),
        np.random.choice(percentage_change_values),
        np.random.choice(ignore_time_interval_before)
    ]


In [32]:
def mutate(individual):
    while True:
        index = random.randint(0, len(individual) - 1)
        if index == 0:  # Mutate rolling window
            individual[index] = np.random.choice(range(5, 49))  # Rolling window from 5H to 48H
            break
        elif index == 1:  # Mutate upper std multiplier
            individual[index] = np.random.choice(np.arange(0.7, 2.1, 0.1))  # Std multipliers from 0.7 to 3 by 0.1
            break
        elif index == 2:  # Mutate lower std multiplier
            individual[index] = np.random.choice(np.arange(0.7, 2.1, 0.1))  # Separate range for lower multiplier
            break
        elif index == 3:  # Mutate TP
            tp = np.random.choice(tp_values)
            if individual[4] < tp:  # Ensure SL < TP
                individual[index] = tp
                break
        elif index == 4:  # Mutate SL
            sl = np.random.choice(sl_values)
            if sl < individual[3]:  # Ensure SL < TP
                individual[index] = sl
                break
        elif index == 5:
            individual[index] = np.random.choice(entry_time_offset_values)
            break
        elif index == 6:
            individual[index] = np.random.choice(percentage_change_values)
            break
        elif index == 7:
            individual[index] = np.random.choice(ignore_time_interval_before)
            break    
    return individual


In [39]:
def evaluate(individual):
    rolling_window, upper_std_multiplier, lower_std_multiplier, tp, sl, entry_time_offset, percentage_change, ignore_time_interval_before = individual
    monthly_rois = []
    penalty_weight = 1
    variance_weight = 1
    

    # Generate new signals with the given rolling window and std multipliers
    signal_data['rolling_mean'] = signal_data['value'].rolling(window=rolling_window).mean()
    signal_data['rolling_std'] = signal_data['value'].rolling(window=rolling_window).std()
    signal_data['upper_bound'] = signal_data['rolling_mean'] + (upper_std_multiplier * signal_data['rolling_std'])
    signal_data['lower_bound'] = signal_data['rolling_mean'] - (lower_std_multiplier * signal_data['rolling_std'])
    
    signal_data['Signal'] = 0
    signal_data.loc[signal_data['value'] > signal_data['upper_bound'], 'Signal'] = -1
    signal_data.loc[signal_data['value'] < signal_data['lower_bound'], 'Signal'] = 1
    
    # Filter the signals as previously done
    filtered_signals = filter_signals(signal_data)
    
    # Iterate over each month from June 2023 to July 2024 and evaluate the strategy
    for year, month_range in [(2023, range(6, 13)), (2024, range(1, 8))]:
        for month in month_range:
            month_signal_data = filtered_signals[
                (filtered_signals['Datetime'].dt.year == year) &
                (filtered_signals['Datetime'].dt.month == month)
            ]

            result = backtest_trades(
                price_data, month_signal_data, tp=tp, sl=sl, 
                entry_time_offset=entry_time_offset, 
                percentage_change=percentage_change, 
                open_order_elimination=120, 
                ignore_time_interval_before=ignore_time_interval_before, 
                ignore_time_interval_after=0
            )

            final_nav = 100000  # Default value if no trades are made
            if not result.empty and 'NAV' in result.columns:
                final_nav = result['NAV'].iloc[-1]
                initial_drawdown = result['Initial Drawdown'].iloc[-1]  # Get the final Initial Drawdown value
            else:
                print(f"Warning: No trades executed for {year}-{month}.")
                initial_drawdown = 0

            roi = ((final_nav - 100000) / 100000) * 100
            monthly_rois.append(roi)
            print(f"  Individual {individual}: {year}-{month} ROI: {roi:.2f}%")
            
            # Check if initial drawdown exceeds the limit
            if initial_drawdown > 3:
                print(f"  Individual {individual}: Initial drawdown exceeded 3% in {year}-{month}. Skipping further evaluation.")
                return -float('inf')  # Assign a very low fitness value

    sum_roi = sum(monthly_rois)  # Ensure this is a list of floats, not tuples
    penalty = sum(penalty_weight * roi for roi in monthly_rois if roi < 0)
    variance_roi = np.var(monthly_rois)

    fitness = sum_roi + penalty - variance_weight * variance_roi
    print(f"Sum of ROIs: {sum_roi:.2f}, Penalty: {penalty:.2f}, Variance: {variance_roi:.4f}, Fitness: {fitness:.4f}")
    return fitness

In [40]:
def simulated_annealing():
    current_individual = create_individual()
    current_fitness = evaluate(current_individual)
    best_individual = list(current_individual)
    best_fitness = current_fitness
    
    initial_temperature = 1.0
    final_temperature = 0.0001
    alpha = 0.99
    temperature = initial_temperature
    
    step = 0
    while temperature > final_temperature:
        step += 1
        new_individual = mutate(list(current_individual))
        new_fitness = evaluate(new_individual)
        
        print(f"Step {step}:")
        print(f"  Current Individual: {current_individual}")
        print(f"  Current Fitness: {current_fitness:.4f}")
        print(f"  New Individual: {new_individual}")
        print(f"  New Fitness: {new_fitness:.4f}")
        print(f"  Temperature: {temperature:.4f}")
        
        if new_fitness > current_fitness or random.uniform(0, 1) < np.exp((new_fitness - current_fitness) / temperature):
            current_individual = new_individual
            current_fitness = new_fitness
        
        if current_fitness > best_fitness:
            best_individual = list(current_individual)
            best_fitness = current_fitness
        
        temperature *= alpha
    
    best_rolling_window, best_upper_std_multiplier, best_lower_std_multiplier, best_tp, best_sl, best_entry_time_offset, best_percentage_change, best_ignore_time_before = best_individual
    optimized_fitness = best_fitness
    
    print(f"\nFinal Optimized Results for June 2023 to July 2024:")
    print(f"  Best Rolling Window: {best_rolling_window} hours")
    print(f"  Best Upper Std Multiplier: {best_upper_std_multiplier}")
    print(f"  Best Lower Std Multiplier: {best_lower_std_multiplier}")
    print(f"  Best Take Profit: {best_tp}")
    print(f"  Best Stop Loss: {best_sl}")
    print(f"  Best Entry Time Offset: {best_entry_time_offset} minutes")
    print(f"  Best Percentage Change: {best_percentage_change}")
    print(f"  Best Ignore Time: {best_ignore_time_before} minutes")
    print(f"  Optimized Fitness (Sum ROI - Variance): {optimized_fitness:.4f}")

def optimize_for_june_2023_to_july_2024():
    simulated_annealing()

# Run the optimization for June 2023 to July 2024
optimize_for_june_2023_to_july_2024()

  Individual [43, 0.7, 0.7, 0.010999999999999998, 0.009999999999999998, 100, 0.0005, 960]: 2023-6 ROI: -3.81%
  Individual [43, 0.7, 0.7, 0.010999999999999998, 0.009999999999999998, 100, 0.0005, 960]: Initial drawdown exceeded 3% in 2023-6. Skipping further evaluation.
  Individual [43, 0.9999999999999999, 0.7, 0.010999999999999998, 0.009999999999999998, 100, 0.0005, 960]: 2023-6 ROI: -3.77%
  Individual [43, 0.9999999999999999, 0.7, 0.010999999999999998, 0.009999999999999998, 100, 0.0005, 960]: Initial drawdown exceeded 3% in 2023-6. Skipping further evaluation.
Step 1:
  Current Individual: [43, 0.7, 0.7, 0.010999999999999998, 0.009999999999999998, 100, 0.0005, 960]
  Current Fitness: -inf
  New Individual: [43, 0.9999999999999999, 0.7, 0.010999999999999998, 0.009999999999999998, 100, 0.0005, 960]
  New Fitness: -inf
  Temperature: 1.0000
  Individual [18, 0.7, 0.7, 0.010999999999999998, 0.009999999999999998, 100, 0.0005, 960]: 2023-6 ROI: -11.79%
  Individual [18, 0.7, 0.7, 0.010999

KeyboardInterrupt: 